# 00 - Smoke test

Comprueba que el entorno (Python, paquetes, dataset y arquitectura CAE-CNNLoc) está correctamente configurado.

Si todas las celdas se ejecutan sin error, ya estamos listos para los notebooks `01_baselines`, `02_cnnloc` y `03_comparativa`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('tensorflow:', tf.__version__)
print('GPUs visible:', tf.config.list_physical_devices('GPU'))

In [ ]:
from cnnloc_rtt.data import load_full_dataset, build_classic_subsets, build_4xN_for_all_subsets

ds = load_full_dataset()
print('APs:', len(ds.aps), '| Reference points:', len(ds.ref_locations))

In [ ]:
classic = build_classic_subsets(ds, window_size=None)
for k, (X, y) in classic.items():
    print(f'{k:25s} -> X={X.shape}, y={y.shape}')

In [ ]:
matrices = build_4xN_for_all_subsets(ds, n_window=16, stride=1)
for k, (X, Y) in matrices.items():
    print(f'{k:25s} -> X={X.shape}, Y={Y.shape}')

In [ ]:
from cnnloc_rtt.models import CAECNNLocConfig, build_autoencoder, build_regressor

cfg = CAECNNLocConfig(n_aps=4, n_window=16)
ae, enc, dec = build_autoencoder(cfg)
ae.summary()
reg = build_regressor(cfg, enc)
reg.summary()

In [ ]:
X_demo, Y_demo = next(iter(matrices.values()))
x_dummy = X_demo[:4]
y_dummy_ae = ae.predict(x_dummy, verbose=0)
y_dummy_reg = reg.predict(x_dummy, verbose=0)
print('AE output shape :', y_dummy_ae.shape)
print('Reg output shape:', y_dummy_reg.shape, '(should be (4, 2))')
assert y_dummy_reg.shape == (4, 2)
print('\nSmoke test passed. You are ready to run 01_baselines.ipynb.')